# Detector panel results — full AI-GenBench set

Five published AI-image detector scores for all **355,638** images in the full benchmark
(175,638 authentic, 180,000 fake = 36 generators × 5,000). This notebook joins them into one
table and looks at what they say.

Nothing here re-runs a detector — it reads `scores/*.csv`, produced on the lab server's GPUs.

**The panel** (five score columns, because DMimageDetection ships two checkpoints):

| column | method | raw score is |
|---|---|---|
| `cnndetection` | ResNet-50 trained on ProGAN (Wang et al. 2020) | pre-sigmoid logit |
| `univfd` | frozen CLIP ViT-L/14 + linear probe (Ojha et al. 2023) | fc logit |
| `dmimagedetection_progan` | ResNet-50, no-downsample stem, GAN-trained (Corvi et al. 2023) | logit, spatial mean |
| `dmimagedetection_latent` | same architecture, latent-diffusion-trained | logit, spatial mean |
| `aeroblade` | autoencoder reconstruction error, training-free (Ricker et al. 2024) | −LPIPS, max over AEs |

All five are oriented the same way: **higher = more synthetic**. None is thresholded — these
are the raw numbers, which is what a likelihood ratio needs.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ROOT   = Path.cwd()
SCORES = ROOT / "scores"

DETECTORS = ["cnndetection", "univfd",
             "dmimagedetection_progan", "dmimagedetection_latent",
             "aeroblade"]

REAL, FAKE = "#3b6ea5", "#c4462c"
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 9, "axes.titlelocation": "left", "axes.titleweight": "bold",
})

## 1. Join the scores

One row per image, one column per detector.

In [ ]:
man = pd.read_csv(ROOT / "manifest.csv")
M = man.copy()

for d in DETECTORS:
    s = pd.read_csv(SCORES / f"{d}.csv")
    assert list(s.columns) == ["image_id", "raw_score"], f"{d}: unexpected columns"
    M = M.merge(s.rename(columns={"raw_score": d}), on="image_id", how="left")

# fail loudly rather than analysing a half-populated table
assert len(M) == len(man), "merge changed the row count"
for d in DETECTORS:
    assert M[d].notna().all(), f"{d}: {M[d].isna().sum()} images unscored"
    assert M[d].std() > 1e-6,  f"{d}: score distribution collapsed"

# provenance: which dataset each authentic image came from
M["source"] = M.image_id.str.split("/").str[0].where(M.label == 0, "fake")
M["release_date"] = pd.to_datetime(M.release_date, errors="coerce")

M.to_csv(SCORES / "master_scores.csv", index=False)
print(f"{len(M)} images x {len(DETECTORS)} detectors -> scores/master_scores.csv")
print(f"labels: {(M.label==0).sum()} authentic / {(M.label==1).sum()} fake")
M.head(3)

## 2. Where the numbers came from

Each score file has a `.meta.json` sidecar recording the exact weights, upstream commit and
library pins that produced it. If two runs disagree, this is what you compare.

In [ ]:
rows = []
for d in DETECTORS:
    m = json.loads((SCORES / f"{d}.meta.json").read_text())
    rows.append({
        "detector": d,
        "score": m["score_semantics"],
        "commit": m["upstream_commit"][:8],
        "crop": m["crop_policy"],
        "torch": m["runtime"]["torch"],
        "device": m["runtime"]["device"],
        "secs": m["elapsed_s"],
    })
prov = pd.DataFrame(rows).set_index("detector")
assert prov.crop.nunique() == 1, "detectors disagree on crop policy"
prov

## 3. Do they separate real from fake?

Each detector on its own scale — the logits and AEROBLADE's LPIPS distance are not comparable
as raw numbers, so don't read across panels here. The dashed line is the authentic median.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(13, 2.4))
for ax, d in zip(axes, DETECTORS):
    a, f = M.loc[M.label == 0, d], M.loc[M.label == 1, d]
    bins = np.linspace(M[d].min(), M[d].max(), 34)
    ax.hist(a, bins=bins, color=REAL, alpha=0.75, label="authentic")
    ax.hist(f, bins=bins, color=FAKE, alpha=0.65, label="fake")
    ax.axvline(a.median(), color=REAL, ls="--", lw=1)
    ax.set_title(d.replace("dmimagedetection_", "dmid_"))
    ax.set_yticks([])
axes[0].set_ylabel("images")
axes[0].legend(frameon=False, fontsize=7.5, loc="upper left")
fig.suptitle("raw score distributions   (higher = more synthetic)", x=0.005, ha="left",
             fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

**Brief.** The fake and authentic curves overlap a lot. That alone does not mean the
detectors fail: each fake curve pools 36 generators across six families (most outside a given
detector's training), and the authentic curve pools four sources. §5 splits by family, §7 by
authentic source. AUROC and overlap summarise the bulk; the evidential question is the upper
tail.

## 4. A common scale

To compare detectors we need one. Every score is restated as **how many standard deviations
above the authentic mean** it sits, using that detector's own authentic images as the
reference. `σ = 0` means "looks like a typical real photo"; large positive means "looks
synthetic to this detector".

This is a reading aid, not a calibrated quantity — the real thing is a likelihood ratio,
fitted later.

In [ ]:
Z = M.copy()
for d in DETECTORS:
    ref = M.loc[M.label == 0, d]
    Z[d] = (M[d] - ref.mean()) / ref.std()

zcols = {d: d.replace("dmimagedetection_", "dmid_") for d in DETECTORS}
print("median σ-above-authentic, by class")
print(Z.groupby(Z.label.map({0: "authentic", 1: "fake"}))[DETECTORS]
       .median().rename(columns=zcols).round(2).to_string())

## 5. Each detector is good at a different thing

AUROC per generator family (authentic images are always the negative class).

**AUROC is the wrong headline statistic for this project** — it is dominated by the bottom of
the fake-score distribution, whereas a likelihood ratio depends on upper-tail discrimination.
It is here as a plumbing check and a rough map of where each detector's competence lies.
Cllr, DET and Tippett come later, on top of `master_scores.csv`.

In [ ]:
def auroc(score, label):
    """Rank-based AUROC; no sklearn in this env."""
    r = pd.Series(score).rank()
    npos, nneg = (label == 1).sum(), (label == 0).sum()
    if npos == 0 or nneg == 0:
        return np.nan
    return (r[label == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg)

fams = [f for f in M.generator_family.unique() if f != "authentic"]
fams = sorted(fams, key=lambda f: -(M.generator_family == f).sum())

tab = pd.DataFrame(index=DETECTORS, columns=["all"] + fams, dtype=float)
for d in DETECTORS:
    tab.loc[d, "all"] = auroc(M[d], M.label)
    for f in fams:
        sub = M[M.generator_family.isin([f, "authentic"])]
        tab.loc[d, f] = auroc(sub[d], sub.label)

counts = {f: int((M.generator_family == f).sum()) for f in fams}
fig, ax = plt.subplots(figsize=(7.2, 2.9))
im = ax.imshow(tab.values.astype(float), cmap="RdYlBu_r",
               norm=TwoSlopeNorm(vmin=0.4, vcenter=0.5, vmax=1.0), aspect="auto")
ax.set_xticks(range(len(tab.columns)))
ax.set_xticklabels([c if c == "all" else f"{c}\nn={counts[c]}" for c in tab.columns], fontsize=8)
ax.set_yticks(range(len(DETECTORS)))
ax.set_yticklabels([zcols[d] for d in DETECTORS], fontsize=8)
ax.grid(False)
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]):
        v = tab.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if (v > 0.85 or v < 0.45) else "black")
ax.set_title("AUROC by generator family  (0.50 = chance)")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
fig.tight_layout()
plt.show()
tab.round(3)

### What that table says

- **The two DMimageDetection checkpoints split the work.** `progan` is the best GAN detector
  (0.98), `latent` the best diffusion detector (0.95). Not redundant.
- **AEROBLADE is at chance on GANs (0.51)** — expected: it tests whether an image reconstructs
  suspiciously well through a *latent-diffusion* autoencoder, which a GAN image has no reason
  to. Its scope is diffusion only; don't pool it naively into a max over all families.
- **CNNDetection (2020) drops from GANs (0.91) to diffusion (0.70)** — a frozen detector
  meeting generators that postdate it.
- Every classifier's overall AUROC is 0.06–0.12 higher than on the 612+612 local sample. Most of
  that is the authentic mix, not the detectors: RAISE, the hardest real source (§7), was a
  third of the local authentic class and is 0.6% of this one.

## 6. Detector competence vs. generator release date

One point per generator (5,000 images each), placed at its release date. Does a frozen panel
get worse on newer generators?

In [ ]:
g = (Z[Z.label == 1]
     .groupby(["generator", "generator_family"], as_index=False)
     .agg(date=("release_date", "first"), **{d: (d, "median") for d in DETECTORS})
     .sort_values("date"))

fig, axes = plt.subplots(1, 5, figsize=(13.5, 2.7), sharey=True)
famcolor = {"gan": "#c4462c", "diffusion": "#3b6ea5", "inpainting": "#7a9a3b",
            "autoregressive": "#b58a2c", "graphics": "#8a6bab", "other": "#888888"}
for ax, d in zip(axes, DETECTORS):
    for fam, sub in g.groupby("generator_family"):
        ax.scatter(sub.date, sub[d], s=18, color=famcolor.get(fam, "#888"), label=fam, zorder=3)
    # least-squares trend across all generators
    x = g.date.map(pd.Timestamp.toordinal).to_numpy(float)
    y = g[d].to_numpy(float)
    b, a = np.polyfit(x, y, 1)
    ax.plot(g.date, a + b * x, color="black", lw=1, ls="--", zorder=2)
    ax.axhline(0, color=REAL, lw=1)
    ax.set_title(f"{zcols[d]}\nslope {b*365:+.2f} σ/yr", fontsize=8)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
axes[0].set_ylabel("σ above authentic")
axes[-1].legend(frameon=False, fontsize=6.5, loc="upper right")
fig.suptitle("per-generator median score vs release date   (blue line = authentic mean)",
             x=0.005, ha="left", fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

**Brief.** Fitted slopes: cnndetection −0.27 σ/yr, univfd −0.23, dmid_progan −0.16,
dmid_latent +0.16, aeroblade +0.09. Read them as descriptive only: release date is confounded
with family (GANs early, diffusion late), and each checkpoint's slope mostly mirrors which
family it was trained on. `temporal_correlation.ipynb` holds family fixed and tests this
properly.

## 7. Not all real images look equally real

The authentic class is 95,158 COCO (train + val) + 79,480 LAION-400M + 1,000 RAISE. COCO and
LAION are web images; RAISE is camera-raw. If provenance shifts the scores of *genuine* images,
the false-positive rate is not a single number — which is the thesis's central open question.

In [ ]:
auth = Z[Z.label == 0]
srcs = ["COCO2017_train", "COCO2017_val", "LAION-400M", "RAISE"]
srcs = [s for s in srcs if (auth.source == s).sum() > 0]

fig, axes = plt.subplots(1, 5, figsize=(13, 2.6), sharey=True)
for ax, d in zip(axes, DETECTORS):
    data = [auth.loc[auth.source == s, d].to_numpy() for s in srcs]
    ax.boxplot(data, tick_labels=[s.replace("COCO2017_", "COCO ").replace("-400M", "") for s in srcs],
               widths=0.55, showfliers=False,
               medianprops=dict(color="black", lw=1.2),
               boxprops=dict(color="#666"), whiskerprops=dict(color="#666"),
               capprops=dict(color="#666"))
    for i, s in enumerate(srcs, start=1):
        v = auth.loc[auth.source == s, d]
        ax.scatter(np.full(len(v), i) + np.random.default_rng(0).uniform(-.12, .12, len(v)),
                   v, s=7, alpha=0.5, color=REAL, zorder=3)
    ax.axhline(Z.loc[Z.label == 1, d].median(), color=FAKE, ls="--", lw=1)
    ax.set_title(zcols[d])
    ax.tick_params(axis="x", labelrotation=35, labelsize=7)
axes[0].set_ylabel("σ above authentic")
fig.suptitle("authentic images only, by source   (red line = median fake)",
             x=0.005, ha="left", fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

print("median σ, authentic images by source")
print(auth.groupby("source")[DETECTORS].median().rename(columns=zcols)
          .reindex(srcs).round(2).to_string())
print("\nn per source:", {s: int((auth.source == s).sum()) for s in srcs})

### Brief: the RAISE gap

**RAISE photos look synthetic to all four classifiers.** They sit +1.1 to +1.8 σ above COCO,
while AEROBLADE (not trained on generator artifacts) runs the other way (−0.15 σ).

| detector | median fake | median RAISE (authentic) |
|---|---|---|
| cnndetection | +1.16 σ | +1.07 σ |
| univfd | +1.23 σ | **+1.66 σ** |
| dmid_progan | +1.47 σ | +1.31 σ |
| dmid_latent | +1.57 σ | +1.11 σ |
| aeroblade | +0.31 σ | −0.15 σ |

On UnivFD the typical genuine camera-raw photo scores *higher* than the typical fake. A
false-positive rate quoted over the mixed authentic class hides this. `crop_confound.ipynb` §4
turns it into rates: at a threshold with 5% false alarms on COCO, 20–58% of RAISE is flagged.

Caveats:
- σ is relative to the authentic mean, now almost all web images (locally one-third RAISE), so
  these σ values are not comparable to the local run's. The rates in `crop_confound` are.
- n = 1,000 is all of RAISE-1k. More camera-raw data needs another source.
- Still confounded with crop scale: a 200×200 crop of a 4928×3264 RAISE frame is a tiny patch,
  while for a ~500 px web image it is most of the frame.

## 8. Do the detectors agree?

Max-over-generators aggregation only buys something if the members are not measuring the same
thing. Spearman correlation of the five score columns, fakes only.

In [ ]:
corr = Z.loc[Z.label == 1, DETECTORS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(4.1, 3.5))
im = ax.imshow(corr.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels([zcols[d] for d in DETECTORS], rotation=40, ha="right", fontsize=7.5)
ax.set_yticks(range(5)); ax.set_yticklabels([zcols[d] for d in DETECTORS], fontsize=7.5)
ax.grid(False)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.values[i, j]) > 0.7 else "black")
ax.set_title("Spearman correlation, fakes only")
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
fig.tight_layout()
plt.show()

**Brief.** Two blocks. cnndetection ↔ dmid_progan ρ = 0.74 (both ProGAN-trained ResNet-50s —
largely redundant), univfd loosely joins them (0.37–0.42); dmid_latent ↔ aeroblade 0.42 (both
key on diffusion). The two blocks are uncorrelated (|ρ| ≤ 0.12), so a max aggregator over them
combines genuinely different signals.

## 9. Play with it

`M` holds the raw scores, `Z` the σ-above-authentic version. Two helpers for looking at actual
images — the fastest way to catch a preprocessing problem that no summary statistic will show.

In [ ]:
from PIL import Image

def view(image_id, ax=None):
    """Show one image's 200x200 crop with all five scores."""
    row = M.loc[M.image_id == image_id].iloc[0]
    zrow = Z.loc[Z.image_id == image_id].iloc[0]
    if ax is None:
        _, ax = plt.subplots(figsize=(2.6, 2.6))
    ax.imshow(Image.open(ROOT / row.crop_path))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    tag = "FAKE" if row.label else "REAL"
    src = row.generator if row.label else row.source
    ax.set_title(f"{tag} · {src}\n" +
                 "  ".join(f"{zcols[d]} {zrow[d]:+.1f}σ" for d in DETECTORS[:2]),
                 fontsize=6.5)
    return ax

def browse(detector, label=0, n=8, most_synthetic=True):
    """Grid of the n images this detector scored highest (or lowest) within one class.

    label=0, most_synthetic=True  -> the panel's worst false positives
    label=1, most_synthetic=False -> the fakes it missed
    """
    sub = Z[Z.label == label].sort_values(detector, ascending=not most_synthetic).head(n)
    ncol = min(n, 4); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.5 * ncol, 2.7 * nrow))
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), sub.iterrows()):
        ax.imshow(Image.open(ROOT / r.crop_path))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        name = r.generator if r.label else r.source
        ax.set_title(f"{name}\n{zcols[detector]} {r[detector]:+.1f}σ", fontsize=7)
    for ax in np.atleast_1d(axes).ravel()[len(sub):]:
        ax.axis("off")
    direction = "most" if most_synthetic else "least"
    fig.suptitle(f"{direction}-synthetic-looking {'authentic' if label==0 else 'fake'} images"
                 f"  ·  by {detector}", x=0.005, ha="left", fontsize=9, fontweight="bold")
    fig.tight_layout()
    return sub[["image_id", "source", "generator", detector]]

The worst false positives — genuine photographs the strongest detector likes least.
(Mostly LAION: it has 79× more images than RAISE, so it dominates any top-8 list. The per-source
*rates* are in `crop_confound.ipynb` §4.)

In [ ]:
browse("dmimagedetection_progan", label=0, n=8, most_synthetic=True)

And the fakes that slipped through — all eight are DeepFloyd IF, exactly as on the local
sample. It is a consistent blind spot of the GAN-trained classifiers.

In [ ]:
browse("dmimagedetection_progan", label=1, n=8, most_synthetic=False)

## Brief: where this leaves things

- **No detector collapsed.** All five give separated, non-degenerate distributions on the full
  set, so the plan (published panel + per-family score models) stands; the embedding fallback
  is not needed.
- **The overlap in §3 is not the right test.** The pooled histograms overlap heavily because
  each "fake" curve mixes 36 generators, most of which a given detector was never built for,
  and the "authentic" curve mixes four sources. Per family (§5) the specialists are strong, and
  a likelihood ratio needs the *upper tail* to separate, not the bulk.
- **The panel is heterogeneous.** cnndetection and dmid_progan are largely redundant (ρ = 0.74);
  dmid_latent and AEROBLADE pair up (0.42); the two pairs are uncorrelated. Same pattern as on
  the local sample.
- **The authentic class is not homogeneous.** Camera-raw RAISE looks as synthetic as the fakes
  to most of the panel. This is the finding with the most consequence for the thesis.

Caveats that travel with every number above:
- **All detectors see 200×200 crops**, not their native resolutions, so every detector gets
  byte-identical pixels and nothing is zero-padded (which would otherwise be a class cue, since
  small images are mostly fakes). This costs fidelity to published benchmark numbers.
- **AEROBLADE ran with 2 of its 3 autoencoders** (SD2's is gated behind Hugging Face auth;
  see `scores/aeroblade.meta.json`).
- **A low score is not evidence of authenticity.** It is equally consistent with degradation,
  or with a generator no panel member has seen.